In [1]:
import polars as pl
from tqdm.auto import tqdm

# Define the desired order of columns for the final DataFrame
COL_ORDERS = ['hash', 'source', 'from', 'to', 'value', 'symbol', 'sub_type', 'transaction_fee', 'type', 'ETH_adj']

# List of datasets to process, each with a path and a unique hash
hashes_to_parse = [
    ["./Data/ABCD 24-03-17 01", "0xe96df8f5ef1a8790415068c798765b07d57643bd"],
   # ["./Data/ABCD 24-04-05 all", "0xf869ee10c36d80e440edabed43926d1065ddef6f"], # we dont have erc_traced in here
    ["./Data/AFY", "0xf43e889444e95a0429c32b0b601dc16edf90fdbf"],
   # ["./Data/ALTN", "0x99de46c4c86f03fed685f8c5ba580aaa80920612"], # we dont have erc_full in here
   # ["./Data/BALD 23-07-30 1j", "0xe96df8f5ef1a8790415068c798765b07d57643bd"], # we do not have internes_formatter.xlsx
   # ["./Data/BALD Full", "0xe96df8f5ef1a8790415068c798765b07d57643bd"], # Repetition from BALD 23-07-30 ? # we do not have internes_formatter.xlsx
   # ["./Data/BLC", "0xd92ed6021b2b21bd5baa51cd1eeedcf7ba2c9312"], # we do not have erc_full in here 
    ["./Data/BRETT 24-03-20 3j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"], # Same as in old dataset ?
    ["./Data/BRETT 24-03-28 2j", "0xBA3F945812a83471d709BCe9C3CA699A19FB46f7"], # Repetition from BRETT 24-03-28 ?gh 
    ["./Data/ECT", "0x5b49001c8b756ab44b6ec3bcd1ea3f3a1c3cd758"],
   # ["./Data/G9", "0x8Bf6267A075f53BcA6168632097ad57707Be475e"], # we do not have erc_full in here
    ["./Data/ORCH", "0x61738277ff9b92e91b8ee02c0b73e8369b79ccfc"],
    ["./Data/PEPE UniV2 24-04-05 all", "0x76fc8dbf2022ee75612cb74ec80caf6b3ccffb23"],
    ["./Data/TOSHI 24-03-24 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"],
    ["./Data/TOSHI 24-04-28 3j", "0x4b0Aaf3EBb163dd45F663b38b6d93f6093EBC2d3"], # Same hash as TOSHI 24-03-24
]

# Iterate over each dataset path and hash pair
for path, src_hash in tqdm(hashes_to_parse):
    # Read the Excel files into Polars DataFrames
    main = pl.read_excel(f"{path}/TX_Main_Pool_or_Add_{src_hash}_formatted.xlsx")
    internal = pl.read_excel(f"{path}/TX_Internes_Pool_or_Add_{src_hash}_formatted.xlsx")
    erc_full = pl.read_excel(f"{path}/TX_ERCFull_Pool_or_Add_{src_hash}_formatted.xlsx")
    erc_traced = pl.read_excel(f"{path}/TX_ERC_Traced_Cleaned_{src_hash}.xlsx")
    erc = pl.read_excel(f"{path}/TX_ERC_Pool_or_Add_{src_hash}_formatted.xlsx") #We have transactionFee in here #new


    # Process the 'main' DataFrame
    main = (
        main
        .select(['hash', 'from', 'to', 'value', 'fee'])
        .with_columns([
            pl.lit("ETH").alias("symbol"),
            pl.lit("main").alias("source"),
            pl.lit("main").alias("sub_type"),
            pl.col("fee").cast(pl.Float64).alias("transaction_fee"), # check this choice
            pl.lit(None).alias("type"),
            pl.lit(None).alias("ETH_adj")
        ])
        .with_columns(pl.col("value").cast(pl.Float64))  # Ensure 'value' is a float
    )

    # Process the 'internal' DataFrame
    internal = (
        internal
        .select(['hash', 'from', 'to', 'value', 'callType'])
        .with_columns([
            pl.lit("ETH").alias("symbol"),
            pl.lit("internal").alias("source"),
            pl.col("callType").alias("sub_type"),
            pl.col("value").cast(pl.Float64).alias("value"),  # Ensure 'value' is a float
            pl.lit(None).alias("transaction_fee"),
            pl.lit(None).alias("type"),
            pl.lit(None).alias("ETH_adj")
        ])
        #.filter(
        #    (pl.col("value") != 0) &  # Compare against numeric zero
        #    (pl.col("sub_type") == "call")
        #)
    )

    # Process the 'erc_full' DataFrame
    erc_full = (
        erc_full
        .select(['hash', 'from', 'to', 'tokenSymbol', 'value', 'tx_type'])
        .with_columns([
            pl.col("tokenSymbol").alias("symbol"),
            pl.col("tx_type").alias("sub_type"),
            pl.lit("erc_full").alias("source"),
            pl.col("value").cast(pl.Float64).alias("value"), 
            pl.lit(None).alias("transaction_fee"),
            pl.lit(None).alias("type"),
            pl.lit(None).alias("ETH_adj")
        ])
    #    .filter(pl.col("value") != 0)  
    )

    # Process ERC_TRACED DataFrame
    erc_traced = (
        erc_traced
        .select(['Tx hash', 'Tx type', 'final_value', 'Fee value'])
        .with_columns([
            pl.col("Tx hash").alias("hash"),
            pl.col("Tx type").alias("type"),
            pl.lit("erc_traced").alias("source"),
            pl.col("final_value").cast(pl.Float64).alias("ETH_adj"),
            pl.col("Fee value").cast(pl.Float64).alias("transaction_fee"),
            pl.lit(None).alias("from"),
            pl.lit(None).alias("to"),
            pl.lit(None).alias("value"),
            pl.lit(None).alias("symbol"),
            pl.lit(None).alias("sub_type"),
        ])
    #    .filter(pl.col("value") != 0)  
    )

###################################################################################### processing an extra data frame:
    # Process ERC DataFrame
    erc = (
        erc
        .select(['hash', 'from', 'to', 'tokenSymbol', 'value', 'tx_type', 'transactionFee'])
        .with_columns([
            pl.col("tokenSymbol").alias("symbol"),
            pl.col("tx_type").alias("sub_type"),
            pl.lit("erc").alias("source"),
            pl.col("value").cast(pl.Float64).alias("value"), 
            pl.col("transactionFee").cast(pl.Float64).alias("transaction_fee"),
            pl.lit(None).alias("type"),
            pl.lit(None).alias("ETH_adj")
        ])
    #    .filter(pl.col("value") != 0)  
    )
########################################################################################################################


    # Add missing columns to each DataFrame to ensure they all have the same columns
    def align_columns(df, cols):
        for col in cols:
            if col not in df.columns:
                df = df.with_columns(pl.lit(None).alias(col))
        return df.select(cols)
    
    main = align_columns(main, COL_ORDERS)
    internal = align_columns(internal, COL_ORDERS)
    erc_full = align_columns(erc_full, COL_ORDERS)
    erc_traced = align_columns(erc_traced, COL_ORDERS)
    erc = align_columns(erc, COL_ORDERS) #new


    # Concatenate DataFrames and process
    df = (
        pl.concat([main, internal, erc_full, erc])
        .sort(["hash", "source", "from", "to"])
    )

    df = (
        df
        .with_columns([
            pl.col("value").cast(pl.Float64),  # Ensure 'value' is a float
            *[pl.col(c).str.to_lowercase().alias(c) for c in ["from", "to"]]
        ])
        #.with_columns(
        #    pl.col("symbol")
        #    .str.replace("WETH", "ETH")
        #    .str.replace("ETH", "WETH")
        #)
    )

    df = df.join(erc_traced, on="hash", how="outer")

    # only the columns I want
    df = df.select(COL_ORDERS)

    # Write to Parquet file
    df.write_parquet(f"./data/{src_hash}.pqt")


  0%|          | 0/9 [00:00<?, ?it/s]/tmp/ipykernel_4860/782707259.py:157: DeprecationWarning: Use of `how='outer'` should be replaced with `how='full'`.
  df = df.join(erc_traced, on="hash", how="outer")
 44%|████▍     | 4/9 [00:44<00:55, 11.07s/it]


KeyboardInterrupt: 

In [2]:
df.schema

Schema([('hash', String),
        ('source', String),
        ('from', String),
        ('to', String),
        ('value', Float64),
        ('symbol', String),
        ('sub_type', String),
        ('transaction_fee', Float64),
        ('type', Null),
        ('ETH_adj', Null)])

In [42]:
df.columns

['hash',
 'source',
 'from',
 'to',
 'value',
 'symbol',
 'sub_type',
 'transaction_fee',
 'type',
 'ETH_adj']

In [21]:
df.head()

hash,source,from,to,value,symbol,sub_type,transaction_fee,type,ETH_adj
str,str,str,str,f64,str,str,f64,null,null
"""0x001197536ee12fd68590b3e69fb0…","""erc""","""0x3fc91a3afd70395cd496c647d5a6…","""0x4b0aaf3ebb163dd45f663b38b6d9…",0.094185,"""WETH""","""ERC20""",0.000021,null,null
"""0x001197536ee12fd68590b3e69fb0…","""erc""","""0x4b0aaf3ebb163dd45f663b38b6d9…","""0x3fc91a3afd70395cd496c647d5a6…",837976.38316,"""TOSHI""","""ERC20""",0.000021,null,null
"""0x001197536ee12fd68590b3e69fb0…","""erc_full""","""0x31225a35fae9e8fdef3fe9cd602d…","""0xfc131b9981fb053c2cab7373daf7…",200.0,"""USDC""","""ERC20""",null,null,null
"""0x001197536ee12fd68590b3e69fb0…","""erc_full""","""0x31225a35fae9e8fdef3fe9cd602d…","""0xd0b53d9277642d899df5c87a3966…",300.0,"""USDC""","""ERC20""",null,null,null
"""0x001197536ee12fd68590b3e69fb0…","""erc_full""","""0x3fc91a3afd70395cd496c647d5a6…","""0x067170777ba8027ced27e034102d…",3494.245367,"""TOSHI""","""ERC20""",null,null,null


In [34]:
import pandas as pd

parquet_path = f"./data/{src_hash}.pqt"

df_polars = pl.read_parquet(parquet_path)

df_pandas = df_polars.to_pandas()

# Exclude 'transaction_fee' column
df_no_fee = df_pandas.drop(columns="transaction_fee")

In [27]:
# Group by all columns except 'transaction_fee' and count occurrences
grouped = df_no_fee.groupby(list(df_no_fee.columns)).size().reset_index(name='count')

# Filter groups with more than 1 occurrence (i.e., duplicates)
duplicates = grouped[grouped['count'] > 1]

# Merge with the original DataFrame to get duplicate rows
df_duplicates = df_pandas.merge(duplicates, on=list(df_no_fee.columns), how='inner')

# To get unique rows (rows without duplicates)
# Using pandas
df_unique = df_pandas[~df_pandas.set_index(list(df_no_fee.columns)).index.isin(
    df_duplicates.set_index(list(df_no_fee.columns)).index
)]


In [28]:
# Convert back to Polars
df_duplicates_polars = pl.DataFrame(df_duplicates)
df_unique_polars = pl.DataFrame(df_unique)

# Save duplicates and unique rows to new Parquet files
df_duplicates_polars.write_parquet(f"./data/{src_hash}_duplicates.pqt")
df_unique_polars.write_parquet(f"./data/{src_hash}_unique.pqt")

In [29]:
# Check if DataFrames are identical using Pandas
are_identical = df_duplicates.equals(df_unique)
print(f"DataFrames are identical: {are_identical}")

DataFrames are identical: False


In [30]:
df_duplicates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   hash             0 non-null      object 
 1   source           0 non-null      object 
 2   from             0 non-null      object 
 3   to               0 non-null      object 
 4   value            0 non-null      float64
 5   symbol           0 non-null      object 
 6   sub_type         0 non-null      object 
 7   transaction_fee  0 non-null      float64
 8   type             0 non-null      object 
 9   ETH_adj          0 non-null      object 
 10  count            0 non-null      int64  
dtypes: float64(2), int64(1), object(8)
memory usage: 124.0+ bytes


In [31]:
df_unique.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79032 entries, 0 to 79031
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   hash             79032 non-null  object 
 1   source           79032 non-null  object 
 2   from             79032 non-null  object 
 3   to               79032 non-null  object 
 4   value            79032 non-null  float64
 5   symbol           79032 non-null  object 
 6   sub_type         79032 non-null  object 
 7   transaction_fee  5961 non-null   float64
 8   type             0 non-null      object 
 9   ETH_adj          0 non-null      object 
dtypes: float64(2), object(8)
memory usage: 6.0+ MB


In [35]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79032 entries, 0 to 79031
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   hash             79032 non-null  object 
 1   source           79032 non-null  object 
 2   from             79032 non-null  object 
 3   to               79032 non-null  object 
 4   value            79032 non-null  float64
 5   symbol           79032 non-null  object 
 6   sub_type         79032 non-null  object 
 7   transaction_fee  5961 non-null   float64
 8   type             0 non-null      object 
 9   ETH_adj          0 non-null      object 
dtypes: float64(2), object(8)
memory usage: 6.0+ MB


In [36]:
print(df.schema)

Schema([('hash', String), ('source', String), ('from', String), ('to', String), ('value', Float64), ('symbol', String), ('sub_type', String), ('transaction_fee', Float64), ('type', Null), ('ETH_adj', Null)])


In [38]:
print(main.schema)
print(internal.schema)
print(erc_full.schema)
print(erc_traced.schema)
print(erc.schema)

Schema([('hash', String), ('source', String), ('from', String), ('to', String), ('value', Float64), ('symbol', String), ('sub_type', String), ('transaction_fee', Float64), ('type', Null), ('ETH_adj', Null)])
Schema([('hash', String), ('source', String), ('from', String), ('to', String), ('value', Float64), ('symbol', String), ('sub_type', String), ('transaction_fee', Null), ('type', Null), ('ETH_adj', Null)])
Schema([('hash', String), ('source', String), ('from', String), ('to', String), ('value', Float64), ('symbol', String), ('sub_type', String), ('transaction_fee', Null), ('type', Null), ('ETH_adj', Null)])
Schema([('hash', String), ('source', String), ('from', Null), ('to', Null), ('value', Null), ('symbol', Null), ('sub_type', Null), ('transaction_fee', Float64), ('type', String), ('ETH_adj', Float64)])
Schema([('hash', String), ('source', String), ('from', String), ('to', String), ('value', Float64), ('symbol', String), ('sub_type', String), ('transaction_fee', Float64), ('type',

In [39]:
print(erc_traced.head())


shape: (5, 10)
┌───────────────┬────────────┬──────┬──────┬───┬──────────┬──────────────┬──────────────┬──────────┐
│ hash          ┆ source     ┆ from ┆ to   ┆ … ┆ sub_type ┆ transaction_ ┆ type         ┆ ETH_adj  │
│ ---           ┆ ---        ┆ ---  ┆ ---  ┆   ┆ ---      ┆ fee          ┆ ---          ┆ ---      │
│ str           ┆ str        ┆ null ┆ null ┆   ┆ null     ┆ ---          ┆ str          ┆ f64      │
│               ┆            ┆      ┆      ┆   ┆          ┆ f64          ┆              ┆          │
╞═══════════════╪════════════╪══════╪══════╪═══╪══════════╪══════════════╪══════════════╪══════════╡
│ 0xfb60dfb8649 ┆ erc_traced ┆ null ┆ null ┆ … ┆ null     ┆ null         ┆ remove       ┆ null     │
│ 5f8c7865516b9 ┆            ┆      ┆      ┆   ┆          ┆              ┆ liquitidy:   ┆          │
│ cdda…         ┆            ┆      ┆      ┆   ┆          ┆              ┆ multicall    ┆          │
│ 0x3ebba6086cc ┆ erc_traced ┆ null ┆ null ┆ … ┆ null     ┆ null         ┆ s